# Algebra 2005–2006 — KC-Expansion Audit

Dataset-specific preprocessing follows pyKT's official Algebra2005 routine:

- question identity = `Problem Name` + `Step Name`;
- drop rows missing student, question, `KC(Default)`, `First Transaction Time`, or label;
- retain binary `Correct First Attempt` labels;
- sort within student by `First Transaction Time`, breaking ties by source-row order;
- split multi-KC assignments on `~~` and retain distinct KCs.

Experimental representations:

- **QL:** one row per source interaction; the lexicographically first KC is the deterministic
  single-KC input, analogous to the lowest-ID rule in the ASSIST2009 notebook.
- **EXP:** one consecutive row per distinct KC assigned to the source interaction.
- **SHUF:** the same EXP row multiset, with non-first KC rows relocated away from their siblings.
- **LEGACY:** the un-deduplicated KC-token expansion used for the ASSIST2009 source-decomposition
  contrast. Equality with EXP is checked and reported rather than assumed.

Required Kaggle input: the raw development file `algebra_2005_2006_train.txt` from the
KDD Cup 2010 / PSLC DataShop release. The notebook also accepts common `.csv`/`.tsv` variants
with the original column names. Do not attach pyKT's already processed `data.txt`, because the
source-row interaction boundary is required for this audit.

Protocol sources: [KDD Cup 2010 / DataShop project](https://pslcdatashop.web.cmu.edu/Project?id=270),
[DataShop tab-delimited schema](https://pslcdatashop.web.cmu.edu/api/DataShop%20Public%20API-v0.41.pdf),
and [pyKT's Algebra2005 preprocessing implementation](https://github.com/pykt-team/pykt-toolkit/blob/main/pykt/preprocess/algebra2005_preprocess.py).



In [2]:
# ================================================================
# 1. CONFIG — exact ASSIST2009 protocol, Algebra2005 dataset
# ================================================================
import os, sys, json, math, glob, time, random, warnings, hashlib, platform
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def find_file(patterns):
    hits = []
    for pattern in patterns:
        hits.extend(glob.glob(pattern))
        hits.extend(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    hits = [h for h in hits if os.path.isfile(h)]
    # Preserve pattern priority, but make duplicate matches deterministic.
    return list(dict.fromkeys(hits))[0] if hits else None

RAW_ALG = find_file([
    "algebra_2005_2006_train.txt",
    "algebra_2005_2006_train.tsv",
    "algebra_2005_2006_train.csv",
    "algebra05_06_train.txt",
    "algebra05-06_train.txt",
    "*algebra*2005*2006*train*.txt",
    "*algebra*2005*train*.csv",
])
assert RAW_ALG, (
    "Algebra2005 raw training file not found. Attach the KDD Cup development dataset "
    "containing algebra_2005_2006_train.txt with the original 19-column schema."
)
print("Algebra2005 raw file:", RAW_ALG)

PIPELINE_VERSION = "v1.1-algebra2005-exact-v3-runplan-2026-07-23"
OUT_DIR = "algebra2005_replication_out"
os.makedirs(OUT_DIR, exist_ok=True)
RESULTS_PATH = f"{OUT_DIR}/algebra2005_replication_results.json"

# Match the ASSIST2009 section flags: A structure, B shortcut, C main
# QL/EXP models, D SHUF, E LEGACY, G aggregation.
RUN = dict(A=True, B=True, C=True, D=True, E=True, G=True)

SEEDS     = [42, 123, 7, 2024, 31]
MAIN_SEED = 42
MIN_SEQ   = 3

# Kept identical to kc-experiment-v3.ipynb.
CFG = dict(
    d_model=128, n_heads=8, n_layers=2, dropout=0.1,
    max_inter=200, train_stride_inter=100, eval_stride_inter=200,
    max_rows=384,
    epochs=60, warmup=3, early_stop=10,
    lr=1e-3, wd=1e-4, label_smooth=0.05,
    ema_decay=0.995, clip=1.0, use_amp=True,
    n_att_bins=16,
)

def split_students(students, seed):
    """80/10/10 student split; identical across all representations."""
    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(students))
    n80, n90 = int(len(idx) * .8), int(len(idx) * .9)
    return idx[:n80], idx[n80:n90], idx[n90:]

def _json_safe(o):
    if isinstance(o, dict): return {str(k): _json_safe(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_json_safe(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return _json_safe(o.tolist())
    return o

if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH, encoding="utf-8") as f:
        _old_results = json.load(f)
    if _old_results.get("pipeline_version") == PIPELINE_VERSION:
        RESULTS = _old_results
        print(f"Resuming: {len(RESULTS.get('runs', {}))} compatible runs in {RESULTS_PATH}")
    else:
        print("Ignoring results from an incompatible pipeline version.")
        RESULTS = {"pipeline_version": PIPELINE_VERSION, "stats": {}, "shortcut": {}, "runs": {}}
else:
    RESULTS = {"pipeline_version": PIPELINE_VERSION, "stats": {}, "shortcut": {}, "runs": {}}

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

RESULTS["provenance"] = {
    "pipeline_version": PIPELINE_VERSION,
    "parent_protocol": "kc-experiment-v3.ipynb / v3.2-canonical-allinone-2026-07-22",
    "dataset": "KDD Cup 2010 Algebra I 2005-2006 development training set",
    "raw_path_basename": os.path.basename(RAW_ALG),
    "raw_size_bytes": os.path.getsize(RAW_ALG),
    "data_sha256": {"Algebra2005": sha256_file(RAW_ALG)},
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "config": dict(CFG),
    "seeds": list(SEEDS),
    "preprocessing_reference": "pykt/preprocess/algebra2005_preprocess.py",
}

def save_results():
    with open(RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump(_json_safe(RESULTS), f, indent=1)


Algebra2005 raw file: /kaggle/input/datasets/youssefamk/algebra2005-dataset/algebra_2005_2006_train.txt


In [3]:
# ================================================================
# 2. DATA PIPELINE — raw Algebra2005 rows → QL / EXP / SHUF
#
# Common schema after materialisation:
#   student_id, question_id, skill, correct, iid, pos
# `iid` is the original source-row interaction index within student.
# Every exploded sibling of one source row shares the same iid and label.
# ================================================================

ALG_COLUMNS = {
    "anon student id": "student_id",
    "problem name": "problem_name",
    "step name": "step_name",
    "first transaction time": "first_transaction_time",
    "correct first attempt": "correct",
    "kc(default)": "kc_default",
}

def read_algebra_table(path):
    wanted = set(ALG_COLUMNS)
    last_error = None
    for encoding in ("utf-8", "latin-1", "ISO-8859-15"):
        for sep in ("\t", ","):
            try:
                df = pd.read_csv(
                    path, sep=sep, encoding=encoding, low_memory=False,
                    usecols=lambda c: c.strip().lower() in wanted,
                )
                normalized = {c.strip().lower() for c in df.columns}
                if wanted.issubset(normalized):
                    return df
            except (UnicodeDecodeError, ValueError, pd.errors.ParserError) as exc:
                last_error = exc
    raise ValueError(
        "Could not read the original Algebra2005 schema. Required columns: "
        + ", ".join(sorted(ALG_COLUMNS))
    ) from last_error

def parse_kc_tokens(value):
    """Sorted nonempty tokens, preserving repeats for the LEGACY condition."""
    parts = [part.strip() for part in str(value).split("~~")]
    return tuple(sorted(part for part in parts if part and part.lower() != "nan"))

def parse_kcs(value):
    """Sorted distinct tokens for QL and clean EXP."""
    return tuple(sorted(set(parse_kc_tokens(value))))

def load_algebra_events(path):
    """pyKT-aligned source-row preprocessing with explicit interaction IDs."""
    df = read_algebra_table(path)
    df = df.rename(columns={c: ALG_COLUMNS[c.strip().lower()] for c in df.columns})
    n_raw = int(len(df))
    df["_source_row"] = np.arange(len(df), dtype=np.int64)

    required = ["student_id", "problem_name", "step_name", "first_transaction_time",
                "correct", "kc_default"]
    df = df.dropna(subset=required).copy()
    n_complete = int(len(df))

    df["correct"] = pd.to_numeric(df["correct"], errors="coerce")
    df = df[df["correct"].isin([0, 1])].copy()
    df["correct"] = df["correct"].astype(np.int8)

    for column in ("student_id", "problem_name", "step_name", "kc_default"):
        df[column] = df[column].astype(str).str.strip()
    df = df[(df["student_id"].str.len() > 0) &
            (df["problem_name"].str.len() > 0) &
            (df["step_name"].str.len() > 0)]

    df["event_time"] = pd.to_datetime(df["first_transaction_time"], errors="coerce")
    n_bad_time = int(df["event_time"].isna().sum())
    df = df.dropna(subset=["event_time"]).copy()
    df["kc_tokens"] = df["kc_default"].map(parse_kc_tokens)
    df["kcs"] = df["kc_default"].map(parse_kcs)
    n_empty_kc = int((df["kcs"].str.len() == 0).sum())
    df = df[df["kcs"].str.len() > 0].copy()

    # pyKT uses Problem Name + Step Name as the question identity.
    df["question_id"] = df["problem_name"] + "----" + df["step_name"]
    df = df.sort_values(
        ["student_id", "event_time", "_source_row"], kind="mergesort"
    ).reset_index(drop=True)
    df["iid"] = df.groupby("student_id", sort=False).cumcount().astype(np.int64)

    # Duplicate tokens inside one KC(Default) cell are removed by parse_kcs.
    duplicate_kc_tokens = int(
        (df["kc_tokens"].str.len() - df["kcs"].str.len()).clip(lower=0).sum()
    )

    audit = {
        "raw_rows_selected_columns": n_raw,
        "rows_complete_required_fields": n_complete,
        "rows_after_binary_time_kc_filters": int(len(df)),
        "dropped_unparseable_time": n_bad_time,
        "dropped_empty_kc_after_split": n_empty_kc,
        "duplicate_kc_tokens_removed_within_source_rows": duplicate_kc_tokens,
    }
    print(
        f"  Algebra2005: {n_raw:,} raw rows -> {len(df):,} source interactions, "
        f"{df['student_id'].nunique():,} students"
    )
    return df[["student_id", "question_id", "correct", "iid", "kcs", "kc_tokens"]], audit

def _with_pos(df):
    df = df.reset_index(drop=True)
    df["pos"] = df.groupby("student_id", sort=False).cumcount().astype(np.int64)
    return df

def build_algebra_conditions(events, load_audit, min_seq=MIN_SEQ):
    """One source row is one interaction; EXP explodes only distinct KCs."""
    ql_full = events.copy()
    counts = ql_full.groupby("student_id", sort=False).size()
    students = sorted(counts[counts >= min_seq].index.astype(str))
    student_set = set(students)
    ql_full = ql_full[ql_full["student_id"].isin(student_set)].reset_index(drop=True)
    ql_full["n_kc"] = ql_full["kcs"].str.len().astype(np.int64)

    exp = (
        ql_full[["student_id", "question_id", "correct", "iid", "kcs"]]
        .explode("kcs")
        .rename(columns={"kcs": "skill"})
        .reset_index(drop=True)
    )
    legacy = (
        ql_full[["student_id", "question_id", "correct", "iid", "kc_tokens"]]
        .explode("kc_tokens")
        .rename(columns={"kc_tokens": "skill"})
        .reset_index(drop=True)
    )
    ql = ql_full[["student_id", "question_id", "correct", "iid", "kcs"]].copy()
    ql["skill"] = ql["kcs"].str[0]  # deterministic analogue of lowest raw KC id

    cols = ["student_id", "question_id", "skill", "correct", "iid"]
    conditions = {
        "QL": _with_pos(ql[cols]),
        "EXP": _with_pos(exp[cols]),
        "LEGACY": _with_pos(legacy[cols]),
    }
    stats = dict(load_audit)
    stats.update({
        "students": int(len(students)),
        "rows_ql": int(len(ql)),
        "rows_exp": int(len(exp)),
        "rows_legacy": int(len(legacy)),
        "avg_kc_per_interaction": float(ql_full["n_kc"].mean()),
        "multi_kc_pct": float((ql_full["n_kc"] > 1).mean() * 100),
        "kc_distribution": {
            str(k): int(v) for k, v in ql_full["n_kc"].value_counts().sort_index().items()
        },
        "unique_questions": int(ql_full["question_id"].nunique()),
        "unique_kcs": int(exp["skill"].nunique()),
        "source_duplicate_condition": "un-deduplicated KC-token expansion",
        "source_duplicate_excess_rows": int(len(legacy) - len(exp)),
        "legacy_equals_exp": bool(len(legacy) == len(exp)),
    })
    stats["expansion_factor_clean"] = stats["rows_exp"] / stats["rows_ql"]
    print(
        f"  [ALG05] students={stats['students']:,} interactions={stats['rows_ql']:,} "
        f"EXP rows={stats['rows_exp']:,}; LEGACY rows={stats['rows_legacy']:,}; "
        f"source-duplicate excess={stats['source_duplicate_excess_rows']:,}; "
        f"avg KCs={stats['avg_kc_per_interaction']:.3f}; "
        f"multi-KC={stats['multi_kc_pct']:.2f}%"
    )
    return conditions, students, stats

def make_shuffled(exp_df, seed=MAIN_SEED):
    """Same EXP rows, but move non-first siblings away from their interaction block."""
    rng = np.random.RandomState(seed)
    cols = ["student_id", "question_id", "skill", "correct", "iid"]
    parts, forced, n_extra = [], 0, 0
    for uid, group in exp_df.groupby("student_id", sort=False):
        arr = group[cols].to_numpy(dtype=object)
        iids = group["iid"].to_numpy()
        first = np.r_[True, iids[1:] != iids[:-1]]
        base = [arr[i] for i in np.where(first)[0]]
        extras = [arr[i] for i in np.where(~first)[0]]
        n_extra += len(extras)
        rng.shuffle(extras)
        for row in extras:
            qid, placed = row[1], False
            for _ in range(64):
                position = rng.randint(0, len(base) + 1)
                left_ok = position == 0 or base[position - 1][1] != qid
                right_ok = position == len(base) or base[position][1] != qid
                if left_ok and right_ok:
                    base.insert(position, row)
                    placed = True
                    break
            if not placed:
                base.insert(rng.randint(0, len(base) + 1), row)
                forced += 1
        parts.append(pd.DataFrame(base, columns=cols))
    out = _with_pos(pd.concat(parts, ignore_index=True))
    same_q = float((out["question_id"] == out.groupby("student_id")["question_id"].shift(1)).mean())
    print(
        f"  [SHUF] relocated {n_extra:,} sibling rows ({forced} forced placements); "
        f"same-question consecutive rate={same_q*100:.2f}%"
    )
    return out

ALG_EVENTS, ALG_LOAD_AUDIT = load_algebra_events(RAW_ALG)
ALG_CONDS, ALG_STUDENTS, ALG_STATS = build_algebra_conditions(ALG_EVENTS, ALG_LOAD_AUDIT)
ALG_CONDS["SHUF"] = make_shuffled(ALG_CONDS["EXP"])

# Executable representation checks. These fail before any GPU time is used.
assert ALG_STATS["multi_kc_pct"] > 0, "Dataset has no multi-KC interactions; wrong file or schema."
assert ALG_STATS["rows_exp"] > ALG_STATS["rows_ql"], "KC expansion did not add rows."
assert not ALG_CONDS["EXP"].duplicated(["student_id", "iid", "skill"]).any()
for name, frame in ALG_CONDS.items():
    assert frame.groupby(["student_id", "iid"])["correct"].nunique().max() == 1, name
    observed = set(map(tuple, frame[["student_id", "iid"]].drop_duplicates().to_numpy()))
    expected = set(map(tuple, ALG_CONDS["QL"][["student_id", "iid"]].to_numpy()))
    assert observed == expected, f"{name}: interaction set differs from QL"

multiset_cols = ["student_id", "question_id", "skill", "correct", "iid"]
exp_sorted = ALG_CONDS["EXP"][multiset_cols].sort_values(multiset_cols).reset_index(drop=True)
shf_sorted = ALG_CONDS["SHUF"][multiset_cols].sort_values(multiset_cols).reset_index(drop=True)
pd.testing.assert_frame_equal(exp_sorted, shf_sorted, check_dtype=False)
legacy_unique = (
    ALG_CONDS["LEGACY"][multiset_cols]
    .drop_duplicates()
    .sort_values(multiset_cols)
    .reset_index(drop=True)
)
pd.testing.assert_frame_equal(exp_sorted, legacy_unique, check_dtype=False)
if ALG_STATS["legacy_equals_exp"]:
    legacy_rows = ALG_CONDS["LEGACY"][multiset_cols].reset_index(drop=True)
    exp_rows = ALG_CONDS["EXP"][multiset_cols].reset_index(drop=True)
    pd.testing.assert_frame_equal(exp_rows, legacy_rows, check_dtype=False)
RESULTS["representation_tests"] = {
    "passed": True,
    "exp_shuf_row_multiset_identical": True,
    "legacy_deduplicates_exactly_to_exp": True,
    "legacy_equals_exp": ALG_STATS["legacy_equals_exp"],
    "identical_interaction_sets": True,
    "one_label_per_interaction": True,
    "no_duplicate_interaction_kc_rows_in_exp": True,
}
print("Representation integrity tests passed.")


  Algebra2005: 809,694 raw rows -> 607,025 source interactions, 574 students
  [ALG05] students=574 interactions=607,025 EXP rows=884,102; LEGACY rows=884,102; source-duplicate excess=0; avg KCs=1.456; multi-KC=31.38%
  [SHUF] relocated 277,077 sibling rows (0 forced placements); same-question consecutive rate=0.18%
Representation integrity tests passed.


In [4]:
# ================================================================
# 3. SECTION A — dataset structure and KC-expansion audit
# ================================================================
def consec_stats(df):
    previous_q = df.groupby("student_id", sort=False)["question_id"].shift(1)
    previous_r = df.groupby("student_id", sort=False)["correct"].shift(1)
    same_q = df["question_id"] == previous_q
    p_q = float(same_q.mean())
    p_l = (
        float((df.loc[same_q, "correct"].to_numpy() == previous_r[same_q].to_numpy()).mean())
        if same_q.any() else float("nan")
    )
    return p_q, p_l

if RUN["A"]:
    print("=" * 72)
    print("SECTION A: Algebra2005 structure")
    print("=" * 72)
    stats = dict(ALG_STATS)
    for condition, frame in ALG_CONDS.items():
        p_q, p_l = consec_stats(frame)
        stats[f"consec_{condition}"] = {
            "rows": int(len(frame)),
            "p_same_q_pct": round(p_q * 100, 4),
            "p_same_label_given_same_q_pct": round(p_l * 100, 4) if not math.isnan(p_l) else None,
        }
        print(
            f"  [ALG05/{condition:4s}] rows={len(frame):>9,}  "
            f"P(same-Q consecutive)={p_q*100:7.3f}%  "
            f"P(same label | same-Q)={'n/a' if math.isnan(p_l) else f'{p_l*100:.3f}%'}"
        )
    print(
        f"  [ALG05/LEGACY] source-duplicate excess="
        f"{stats['source_duplicate_excess_rows']:,} rows; "
        f"LEGACY equals EXP={stats['legacy_equals_exp']}"
    )
    RESULTS["stats"]["ALG05"] = stats
    save_results()

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    distribution = stats["kc_distribution"]
    ks = [int(k) for k in distribution]
    values = [distribution[str(k)] / sum(distribution.values()) * 100 for k in ks]
    axes[0].bar(ks, values, color="#2196F3")
    for k, value in zip(ks, values):
        if value >= 0.2:
            axes[0].text(k, value + 0.3, f"{value:.1f}%", ha="center", fontsize=8)
    axes[0].set_xlabel("Distinct KCs per source interaction")
    axes[0].set_ylabel("% of interactions")
    axes[0].set_title(
        f"Algebra2005 KC distribution\n({stats['multi_kc_pct']:.1f}% multi-KC; "
        f"mean {stats['avg_kc_per_interaction']:.3f})"
    )

    names = ["QL\n(source interactions)", "EXP\n(distinct-KC rows)",
             "LEGACY\n(raw KC tokens)"]
    rows = [stats["rows_ql"], stats["rows_exp"], stats["rows_legacy"]]
    colors = ["#1976D2", "#FF9800", "#B71C1C"]
    bars = axes[1].bar(names, rows, color=colors)
    for bar, value in zip(bars, rows):
        axes[1].text(bar.get_x() + bar.get_width()/2, value * 1.01, f"{value:,}",
                     ha="center", fontweight="bold")
    axes[1].set_ylabel("model rows")
    axes[1].set_title(
        f"Clean expansion: ×{stats['expansion_factor_clean']:.3f}; "
        f"source excess: {stats['source_duplicate_excess_rows']:,}"
    )

    conditions = ["QL", "EXP", "LEGACY", "SHUF"]
    pq = [stats[f"consec_{c}"]["p_same_q_pct"] for c in conditions]
    pl = [stats[f"consec_{c}"]["p_same_label_given_same_q_pct"] or 0 for c in conditions]
    x = np.arange(len(conditions)); width = 0.36
    axes[2].bar(x - width/2, pq, width, color="#2196F3", label="P(same-Q consecutive)")
    axes[2].bar(x + width/2, pl, width, color="#FF5722", label="P(same label | same-Q)")
    axes[2].set_xticks(x); axes[2].set_xticklabels(conditions)
    axes[2].set_ylim(0, 110); axes[2].legend(fontsize=9)
    axes[2].set_title("Consecutive-row shortcut structure")
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/alg_fig1_dataset_structure.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved: {OUT_DIR}/alg_fig1_dataset_structure.png")


SECTION A: Algebra2005 structure
  [ALG05/QL  ] rows=  607,025  P(same-Q consecutive)=  0.365%  P(same label | same-Q)=74.684%
  [ALG05/EXP ] rows=  884,102  P(same-Q consecutive)= 31.591%  P(same label | same-Q)=99.799%
  [ALG05/LEGACY] rows=  884,102  P(same-Q consecutive)= 31.591%  P(same label | same-Q)=99.799%
  [ALG05/SHUF] rows=  884,102  P(same-Q consecutive)=  0.178%  P(same label | same-Q)=75.222%
  [ALG05/LEGACY] source-duplicate excess=0 rows; LEGACY equals EXP=True
Saved: algebra2005_replication_out/alg_fig1_dataset_structure.png


In [5]:
# ================================================================
# 4. SECTION B — zero-parameter shortcut baseline
# Identical rule and test-population evaluation to ASSIST2009:
# for each target row, search the preceding five rows from the same
# student and copy the most recent same-question label when present.
# Otherwise predict the student's running correctness mean. Row zero is
# context-only, so the defined 0.5 cold-start fallback is never evaluated.
# ================================================================
from sklearn.metrics import roc_auc_score

def compute_auc(y_true, y_pred):
    y_true = np.asarray(y_true, int)
    y_pred = np.asarray(y_pred, float)
    if y_true.size == 0 or len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_pred))

def shortcut_auc(df, students_subset=None, lookback=5):
    if students_subset is not None:
        df = df[df["student_id"].isin(set(students_subset))]
    y_true, y_pred, fires = [], [], 0
    for _, group in df.groupby("student_id", sort=False):
        questions = group["question_id"].to_numpy()
        responses = group["correct"].to_numpy()
        running_correct, running_total = int(responses[0]), 1
        for i in range(1, len(questions)):
            prediction = None
            for j in range(i - 1, max(-1, i - 1 - lookback), -1):
                if questions[j] == questions[i]:
                    prediction = float(responses[j])
                    fires += 1
                    break
            if prediction is None:
                prediction = running_correct / running_total if running_total else 0.5
            y_true.append(int(responses[i]))
            y_pred.append(prediction)
            running_correct += int(responses[i])
            running_total += 1
    return compute_auc(y_true, y_pred), fires / max(len(y_true), 1)

if RUN["B"]:
    print("=" * 72)
    print("SECTION B: Shortcut baseline")
    print("=" * 72)
    cases = [(f"ALG05/{condition}", ALG_CONDS[condition], ALG_STUDENTS)
             for condition in ("QL", "EXP", "LEGACY", "SHUF")]
    for name, frame, students in cases:
        if name in RESULTS["shortcut"]:
            result = RESULTS["shortcut"][name]
            print(f"  [{name}] cached: AUC={result['auc_mean']:.4f}")
            continue
        aucs, fire_rates = [], []
        for seed in SEEDS:
            _, _, test_idx = split_students(students, seed)
            test_students = [students[i] for i in test_idx]
            auc, fire_rate = shortcut_auc(frame, test_students)
            aucs.append(auc); fire_rates.append(fire_rate)
        RESULTS["shortcut"][name] = {
            "auc_per_seed": aucs,
            "fire_per_seed": fire_rates,
            "auc_mean": float(np.mean(aucs)),
            "auc_std": float(np.std(aucs, ddof=1)),
            "fire_mean": float(np.mean(fire_rates)),
        }
        save_results()
        print(
            f"  [{name:10s}] AUC={np.mean(aucs):.4f}±{np.std(aucs, ddof=1):.4f}  "
            f"fire={np.mean(fire_rates)*100:5.1f}%"
        )

    labels = [name.split("/")[1] for name, _, _ in cases]
    aucs = [RESULTS["shortcut"][name]["auc_mean"] for name, _, _ in cases]
    errors = [RESULTS["shortcut"][name]["auc_std"] for name, _, _ in cases]
    fires = [RESULTS["shortcut"][name]["fire_mean"] * 100 for name, _, _ in cases]
    fig, ax1 = plt.subplots(figsize=(9.5, 5))
    x = np.arange(len(labels))
    bars = ax1.bar(x, aucs, 0.55, yerr=errors, capsize=4,
                   color=["#4CAF50", "#F44336", "#B71C1C", "#FF9800"])
    ax1.axhline(0.5, color="gray", ls="--", lw=0.8)
    ax1.set_xticks(x); ax1.set_xticklabels(labels)
    ax1.set_ylabel("AUC"); ax1.set_ylim(0.45, 1.0)
    ax1.set_title("Algebra2005 shortcut baseline\ntest students; mean ± sample sd over 5 seeds")
    for bar, value in zip(bars, aucs):
        ax1.text(bar.get_x() + bar.get_width()/2, value + 0.006, f"{value:.3f}",
                 ha="center", fontweight="bold")
    ax2 = ax1.twinx(); ax2.plot(x, fires, "ko--", ms=6); ax2.set_ylabel("fire rate (%)")
    for xi, fire_rate in zip(x, fires):
        ax2.annotate(f"{fire_rate:.1f}%", (xi, fire_rate),
                     textcoords="offset points", xytext=(8, 4), fontsize=8)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/alg_fig2_shortcut_baseline.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved: {OUT_DIR}/alg_fig2_shortcut_baseline.png")


SECTION B: Shortcut baseline
  [ALG05/QL  ] AUC=0.6192±0.0245  fire=  1.6%
  [ALG05/EXP ] AUC=0.8354±0.0106  fire= 32.1%
  [ALG05/LEGACY] AUC=0.8354±0.0106  fire= 32.1%
  [ALG05/SHUF] AUC=0.6157±0.0238  fire=  1.1%
Saved: algebra2005_replication_out/alg_fig2_shortcut_baseline.png


In [6]:
# ================================================================
# 5. MODELS + TRAINING INFRASTRUCTURE  (v3)
#
# Windows are built over INTERACTIONS (never split), so all conditions
# see the same history coverage per window.
#
# Evaluation protocols:
#   standard   : teacher-forced row inputs (leaky on expanded data);
#                yields row AUC and fused (interaction-averaged) AUC.
#   all-in-one : every KC of an interaction is predicted independently
#                with history ending at the previous interaction, while
#                PAST interactions keep their real responses in context
#                (faithful history, as in pyKT's protocol). Exact for all
#                three models:
#                - attention models: each response is embedded on its
#                  source row, attention is strictly causal, and every
#                  same-interaction key/value path is banned. Queries
#                  carry no response; all prior interactions remain.
#                - DKT: every sibling is read directly from the recurrent
#                  state at the end of the previous interaction.
# Model selection: validation all-in-one fused AUC for every run
# (coincides with row AUC on question-level data) — no leaky selection.
# ================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
RESULTS["provenance"]["torch"] = torch.__version__

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    print("WARNING: no GPU found — training will be extremely slow.")

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def deterministic_log_bin(arr, n_bins):
    """History-only count bins; no quantile is fitted on validation/test users."""
    arr = np.clip(np.asarray(arr, float), 0, None)
    return np.minimum(np.floor(np.log2(arr + 1)).astype(np.int64), n_bins - 1)

def prepare_condition(df, student_index):
    """Encode ids; sequences keyed by encoded student.
    Array columns: [u, q, s, r, att_bin, iid]."""
    df = df.copy()
    df["u"] = df["student_id"].map(student_index).astype(np.int64)
    df["q"] = pd.factorize(df["question_id"].astype(str))[0].astype(np.int64)
    df["s"] = pd.factorize(df["skill"].astype(str))[0].astype(np.int64)
    nq, ns = int(df["q"].max()) + 1, int(df["s"].max()) + 1
    df = df.sort_values(["u", "pos"], kind="mergesort").reset_index(drop=True)
    # Opportunity counts advance once per distinct interaction-KC, not once
    # per duplicate record; every duplicate of an interaction-KC shares a bin.
    opp = df[["u", "iid", "s"]].drop_duplicates().copy()
    opp["att_count"] = opp.groupby(["u", "s"], sort=False).cumcount()
    df = df.merge(opp, on=["u", "iid", "s"], how="left", sort=False, validate="many_to_one")
    df = df.sort_values(["u", "pos"], kind="mergesort").reset_index(drop=True)
    att = deterministic_log_bin(df["att_count"].to_numpy(float), CFG["n_att_bins"])
    arr = np.stack([df["u"].to_numpy(np.int32), df["q"].to_numpy(np.int32),
                    df["s"].to_numpy(np.int32), df["correct"].to_numpy(np.int32),
                    att.astype(np.int32), df["iid"].to_numpy(np.int32)], axis=1)
    u_arr  = arr[:, 0]
    bounds = np.where(np.diff(u_arr))[0] + 1
    seqs = {int(uid): seg for uid, seg in zip(u_arr[np.r_[0, bounds]], np.split(arr, bounds))}
    return seqs, nq, ns

def build_window_catalog(conditions, student_index, tag):
    """Build shared windows from QL interaction IDs before model-row materialisation.
    The row budget is computed from the maximum per-interaction row count over
    all conditions, so any shortening is identical and no interaction is split."""
    ql = conditions["QL"][["student_id", "iid", "pos"]].copy()
    ql["u"] = ql["student_id"].map(student_index).astype(np.int64)
    ordered = {int(u): g.sort_values("pos")["iid"].astype(int).tolist()
               for u, g in ql.groupby("u", sort=False)}
    max_cost = {}
    for cdf in conditions.values():
        z = cdf[["student_id", "iid"]].copy()
        z["u"] = z["student_id"].map(student_index).astype(np.int64)
        for (u, iid), n in z.groupby(["u", "iid"], sort=False).size().items():
            key = (int(u), int(iid)); max_cost[key] = max(max_cost.get(key, 0), int(n))

    def make_specs(stride):
        out, shortened, shared_rows = {}, 0, []
        for u, iids in ordered.items():
            costs = np.array([max_cost[(u, iid)] for iid in iids], dtype=np.int64)
            if costs.max(initial=0) > CFG["max_rows"]:
                raise ValueError(f"{tag}: one interaction exceeds max_rows; increase the shared budget")
            pref = np.r_[0, np.cumsum(costs)]; n_int = len(iids); s0 = 0
            while s0 < n_int:
                nominal = min(s0 + CFG["max_inter"], n_int); e0 = nominal
                while e0 - s0 > 1 and pref[e0] - pref[s0] > CFG["max_rows"]:
                    e0 -= 1
                if pref[e0] - pref[s0] > CFG["max_rows"]:
                    raise ValueError(f"{tag}: shared window budget cannot hold interaction {iids[s0]}")
                if e0 - s0 >= 2:
                    out.setdefault(u, []).append(tuple(iids[s0:e0]))
                    shared_rows.append(int(pref[e0] - pref[s0]))
                    shortened += int(e0 < nominal)
                if e0 >= n_int: break
                s0 += min(stride, max(1, e0 - s0))
        audit = {"windows": int(sum(map(len, out.values()))),
                 "shortened_by_shared_row_budget": int(shortened),
                 "max_shared_rows": int(max(shared_rows, default=0)),
                 "max_interactions": int(max((len(w) for ws in out.values() for w in ws), default=0))}
        return out, audit

    tr, tr_a = make_specs(CFG["train_stride_inter"])
    ev, ev_a = make_specs(CFG["eval_stride_inter"])
    print(f"  [{tag}] shared windows: train={tr_a}, eval={ev_a}")
    return {"train": tr, "eval": ev, "audit": {"train": tr_a, "eval": ev_a}}

class KTDataset(Dataset):
    """Materialise a condition using shared original-interaction windows."""
    def __init__(self, seqs, specs, users):
        self.s = []
        for u in users:
            u = int(u)
            if u not in seqs: continue
            seq = seqs[u]
            for spec in specs.get(u, []):
                w = seq[np.isin(seq[:, 5], np.asarray(spec, dtype=seq.dtype))]
                if set(map(int, np.unique(w[:, 5]))) != set(map(int, spec)):
                    raise AssertionError(f"condition is missing interaction(s) for user {u}")
                target = (w[:, 5] != int(spec[0])).astype(np.int64)
                self.s.append(tuple(w[:, c].astype(np.int64) for c in range(6)) + (target,))
    def __len__(self): return len(self.s)
    def __getitem__(self, i): return self.s[i]

def make_collate(pad_q, pad_s):
    def cfn(batch):
        L = max(len(x[0]) for x in batch); B = len(batch)
        u_ = np.zeros((B, L), np.int64); q_ = np.full((B, L), pad_q, np.int64)
        s_ = np.full((B, L), pad_s, np.int64); r_ = np.zeros((B, L), np.int64)
        a_ = np.zeros((B, L), np.int64); i_ = np.full((B, L), -1, np.int64)
        m_ = np.zeros((B, L), np.float32); z_ = np.zeros((B, L), np.bool_)
        for i, (u, q, s, r, a, ii, z) in enumerate(batch):
            l = len(q)
            u_[i, :l] = u; q_[i, :l] = q; s_[i, :l] = s
            r_[i, :l] = r; a_[i, :l] = a; i_[i, :l] = ii; m_[i, :l] = 1.; z_[i, :l] = z
        t = lambda x, d: torch.tensor(x, dtype=d)
        return (t(u_, torch.long), t(q_, torch.long), t(s_, torch.long),
                t(r_, torch.long), t(a_, torch.long), t(i_, torch.long),
                t(m_, torch.float32), t(z_, torch.bool))
    return cfn

def make_loaders(seqs, students, nq, ns, seed, window_catalog):
    tr_i, va_i, te_i = split_students(students, seed)
    if torch.cuda.is_available():
        ng = torch.cuda.device_count()
        vr = torch.cuda.get_device_properties(0).total_memory / 1e9
        bs = (64 if vr >= 14 else 32) * ng
    else:
        bs = 16
    cfn = make_collate(nq, ns)
    nw  = 2 if torch.cuda.is_available() else 0
    def mk(users, shuffle):
        specs = window_catalog["train" if shuffle else "eval"]
        return DataLoader(KTDataset(seqs, specs, users),
                          bs, shuffle=shuffle, num_workers=nw,
                          pin_memory=torch.cuda.is_available(), collate_fn=cfn)
    return mk(tr_i, True), mk(va_i, False), mk(te_i, False)

# ── models: forward(q, s, r, a, mask, ban=None) ────────────────
# ban: (B, L, L) bool, True = attention prohibited (same-interaction
# rows). Attention is already strictly causal, including the diagonal.
class DKT(nn.Module):
    def __init__(self, nq, ns, d=128, dr=0.1, **kw):
        super().__init__()
        self.emb = nn.Embedding(2 * (ns + 1), d)
        self.gru  = nn.GRU(d, d, batch_first=True)
        self.drop = nn.Dropout(dr)
        self.head = nn.Linear(d, ns + 1)
        for m in self.modules():
            if isinstance(m, nn.Linear): nn.init.xavier_uniform_(m.weight)
            elif isinstance(m, nn.Embedding): nn.init.normal_(m.weight, std=0.01)
    def _readout(self, h, s):
        return self.head(self.drop(h)).gather(-1, s.unsqueeze(-1)).squeeze(-1)
    def forward(self, q, s, r, a, mask, ban=None):
        # Canonical DKT timing: update with (skill_t, response_t), then shift
        # the hidden states so the prediction for t uses history through t-1.
        x = self.emb((s * 2 + r).clamp(max=self.emb.num_embeddings - 1))
        h_after, _ = self.gru(self.drop(x))
        h_before = torch.zeros_like(h_after); h_before[:, 1:] = h_after[:, :-1]
        return self._readout(h_before, s)
    def allinone_logits(self, q, s, r, a, mask, ii):
        """Every sibling is read from the same state at the previous
        interaction boundary; no target-interaction update precedes prediction."""
        B, L = r.shape; d = self.gru.hidden_size
        x = self.emb((s * 2 + r).clamp(max=self.emb.num_embeddings - 1))
        h, _ = self.gru(self.drop(x))                     # observed past interactions
        fr = torch.ones_like(ii, dtype=torch.bool)
        fr[:, 1:] = ii[:, 1:] != ii[:, :-1]                # first row of interaction
        idx = torch.arange(L, device=r.device).unsqueeze(0).expand(B, L)
        first_idx = torch.cummax(torch.where(fr, idx, torch.zeros_like(idx)), 1).values
        prev_end = first_idx - 1                           # -1 -> window-initial state
        h_prev = h.gather(1, prev_end.clamp(min=0).unsqueeze(-1).expand(B, L, d))
        h_prev = torch.where((prev_end >= 0).unsqueeze(-1), h_prev, torch.zeros_like(h_prev))
        return self._readout(h_prev, s)

class _AKTAttn(nn.Module):
    def __init__(self, d, nh, dr):
        super().__init__()
        self.H = nh; self.dk = d // nh
        self.Wq = nn.Linear(d, d); self.Wk = nn.Linear(d, d)
        self.Wv = nn.Linear(d, d); self.Wo = nn.Linear(d, d); self.drop = nn.Dropout(dr)
        self.gamma = nn.Parameter(torch.full((nh,), math.log(math.e - 1)))
    def forward(self, Q, K, V, mask=None, ban=None):
        B, L, D = Q.shape; H, dk = self.H, self.dk
        sp = lambda x: x.view(B, L, H, dk).transpose(1, 2)
        Qp, Kp, Vp = sp(self.Wq(Q)), sp(self.Wk(K)), sp(self.Wv(V))
        sc = (Qp @ Kp.transpose(-2, -1)) / math.sqrt(dk)
        t  = torch.arange(L, device=Q.device, dtype=torch.float)
        dist = (t.unsqueeze(1) - t.unsqueeze(0)).clamp(0)
        sc = sc * torch.exp(-F.softplus(self.gamma).view(1, H, 1, 1) * dist.unsqueeze(0).unsqueeze(0))
        cm = torch.triu(torch.ones(L, L, device=Q.device, dtype=torch.bool), 0)
        sc = sc.masked_fill(cm.unsqueeze(0).unsqueeze(0), float("-inf"))
        if mask is not None: sc = sc.masked_fill((mask == 0).unsqueeze(1).unsqueeze(2), float("-inf"))
        if ban is not None:  sc = sc.masked_fill(ban.unsqueeze(1), float("-inf"))
        at = torch.nan_to_num(F.softmax(sc, -1), nan=0.)
        return self.Wo((self.drop(at) @ Vp).transpose(1, 2).contiguous().view(B, L, D))

class _PlainAttn(nn.Module):
    def __init__(self, d, nh, dr):
        super().__init__()
        self.H = nh; self.dk = d // nh
        self.Wq = nn.Linear(d, d); self.Wk = nn.Linear(d, d)
        self.Wv = nn.Linear(d, d); self.Wo = nn.Linear(d, d); self.drop = nn.Dropout(dr)
    def forward(self, Q, K, V, mask=None, ban=None):
        B, L, D = Q.shape; H, dk = self.H, self.dk
        sp = lambda x: x.view(B, L, H, dk).transpose(1, 2)
        Qp, Kp, Vp = sp(self.Wq(Q)), sp(self.Wk(K)), sp(self.Wv(V))
        sc = (Qp @ Kp.transpose(-2, -1)) / math.sqrt(dk)
        cm = torch.triu(torch.ones(L, L, device=Q.device, dtype=torch.bool), 0)
        sc = sc.masked_fill(cm.unsqueeze(0).unsqueeze(0), float("-inf"))
        if mask is not None: sc = sc.masked_fill((mask == 0).unsqueeze(1).unsqueeze(2), float("-inf"))
        if ban is not None:  sc = sc.masked_fill(ban.unsqueeze(1), float("-inf"))
        at = torch.nan_to_num(F.softmax(sc, -1), nan=0.)
        return self.Wo((self.drop(at) @ Vp).transpose(1, 2).contiguous().view(B, L, D))

def _blk(Attn, d, nh, dr):
    class Blk(nn.Module):
        def __init__(self):
            super().__init__()
            self.attn = Attn(d, nh, dr)
            self.ff = nn.Sequential(nn.Linear(d, d * 4), nn.GELU(), nn.Dropout(dr), nn.Linear(d * 4, d))
            self.n1 = nn.LayerNorm(d); self.n2 = nn.LayerNorm(d); self.drop = nn.Dropout(dr)
        def forward(self, q, kv, mask=None, ban=None):
            x = q + self.drop(self.attn(self.n1(q), self.n1(kv), self.n1(kv), mask, ban))
            return x + self.drop(self.ff(self.n2(x)))
    return Blk()

class AKTR(nn.Module):
    def __init__(self, nq, ns, d=128, nh=8, nl=2, dr=0.1, n_att=16):
        super().__init__()
        self.kc = nn.Embedding(ns + 1, d); self.dd = nn.Embedding(ns + 1, d)
        self.mu = nn.Embedding(nq + 1, 1); self.re = nn.Embedding(2, d)
        self.ae = nn.Embedding(n_att, d)
        self.q_enc = nn.ModuleList([_blk(_AKTAttn, d, nh, dr) for _ in range(nl)])
        self.k_enc = nn.ModuleList([_blk(_AKTAttn, d, nh, dr) for _ in range(nl)])
        self.ret   = nn.ModuleList([_blk(_AKTAttn, d, nh, dr) for _ in range(nl)])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(d, d // 2), nn.GELU(), nn.Dropout(dr), nn.Linear(d // 2, 1))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding): nn.init.normal_(m.weight, std=0.01)
        nn.init.zeros_(self.mu.weight)
    def forward(self, q, s, r, a, mask, ban=None):
        xq = self.kc(s) + self.mu(q) * self.dd(s); xi = xq + self.re(r) + self.ae(a)
        h = xq
        for blk in self.q_enc: h = blk(h, h, mask, ban)
        k = xi
        for blk in self.k_enc: k = blk(k, k, mask, ban)
        for blk in self.ret:   h = blk(h, k, mask, ban)
        return self.head(self.norm(h)).squeeze(-1)

class SimpleKT(nn.Module):
    def __init__(self, nq, ns, d=128, nh=8, nl=2, dr=0.1, n_att=16):
        super().__init__()
        self.kc = nn.Embedding(ns + 1, d); self.var = nn.Embedding(ns + 1, d)
        self.dif = nn.Embedding(nq + 1, d); self.res = nn.Embedding(2, d)
        self.ae = nn.Embedding(n_att, d)
        self.blocks = nn.ModuleList([_blk(_PlainAttn, d, nh, dr) for _ in range(nl)])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(d, d // 2), nn.GELU(), nn.Dropout(dr), nn.Linear(d // 2, 1))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding): nn.init.normal_(m.weight, std=0.01)
        nn.init.zeros_(self.dif.weight)
    def forward(self, q, s, r, a, mask, ban=None):
        x = self.kc(s) + self.dif(q) * self.var(s); y = x + self.res(r) + self.ae(a)
        h = x
        for blk in self.blocks: h = blk(h, y, mask, ban)
        return self.head(self.norm(h)).squeeze(-1)

def build_model(name, nq, ns):
    d, nh, nl = CFG["d_model"], CFG["n_heads"], CFG["n_layers"]
    dr, na = CFG["dropout"], CFG["n_att_bins"]
    if name == "DKT":      return DKT(nq, ns, d, dr)
    if name == "AKT-R":    return AKTR(nq, ns, d, nh, nl, dr, na)
    if name == "simpleKT": return SimpleKT(nq, ns, d, nh, nl, dr, na)
    raise ValueError(name)

class EMA:
    def __init__(self, m, d):
        self.d = d
        self.s = {n: p.data.clone() for n, p in m.named_parameters() if p.requires_grad}
        self.b = {}
    @torch.no_grad()
    def update(self, m):
        for n, p in m.named_parameters():
            if p.requires_grad: self.s[n].mul_(self.d).add_(p.data, alpha=1 - self.d)
    def apply(self, m):
        self.b = {}
        for n, p in m.named_parameters():
            if p.requires_grad: self.b[n] = p.data.clone(); p.data.copy_(self.s[n])
    def restore(self, m):
        for n, p in m.named_parameters():
            if p.requires_grad: p.data.copy_(self.b[n])
        self.b = {}

def _scaler(en):
    try: return torch.amp.GradScaler("cuda", enabled=en)
    except Exception: return torch.cuda.amp.GradScaler(enabled=en)
def _autocast(en):
    try: return torch.amp.autocast("cuda", enabled=en)
    except Exception: return torch.cuda.amp.autocast(enabled=en)

def cosine_lr(opt, warm, total):
    def fn(s):
        if s < warm: return s / max(1, warm)
        return max(0., .5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    return torch.optim.lr_scheduler.LambdaLR(opt, fn)

def crit_fn(sm):
    def f(lg, t): return F.binary_cross_entropy_with_logits(
        lg, t.float() * (1 - sm) + .5 * sm, reduction="none")
    return f

def allinone_ban(ii):
    """Ban every key/value row from the query's own interaction.
    Responses are attached to their source rows and the base attention mask
    is strictly causal, so all earlier interactions remain available."""
    return ii.unsqueeze(2) == ii.unsqueeze(1)

@torch.no_grad()
def run_protocol_smoke_tests():
    """Non-trivial protocol tests: sever target labels, retain prior history."""
    dev = torch.device(DEVICE); torch.manual_seed(1701)
    q = torch.tensor([[0, 1, 2, 3, 1]], device=dev)
    s = torch.tensor([[0, 1, 2, 3, 1]], device=dev)
    r = torch.tensor([[1, 0, 0, 1, 1]], device=dev)
    a = torch.zeros_like(s); m = torch.ones_like(s, dtype=torch.float32)
    ii = torch.tensor([[0, 1, 1, 2, 2]], device=dev)
    target = (ii == 1)
    models = [("DKT", DKT(5, 5, d=32, dr=0.0)),
              ("AKT-R", AKTR(5, 5, d=32, nh=4, nl=2, dr=0.0, n_att=4)),
              ("simpleKT", SimpleKT(5, 5, d=32, nh=4, nl=2, dr=0.0, n_att=4))]
    def ai(model, rr):
        return (model.allinone_logits(q, s, rr, a, m, ii) if hasattr(model, "allinone_logits")
                else model(q, s, rr, a, m, allinone_ban(ii)))
    report = {}; max_target_delta = 0.0
    for name, model in models:
        model = model.to(dev).eval(); base = ai(model, r)
        for j in torch.where(target[0])[0].tolist():
            changed = r.clone(); changed[0, j] = 1 - changed[0, j]
            changed_pred = ai(model, changed)[target]
            max_target_delta = max(max_target_delta, float((base[target] - changed_pred).abs().max()))
            torch.testing.assert_close(base[target], changed_pred, rtol=0, atol=1e-7)
        prior = r.clone(); prior[0, 0] = 1 - prior[0, 0]
        history_delta = (base[target] - ai(model, prior)[target]).abs()
        assert torch.all(history_delta > 1e-8), f"{name}: a sibling lost preceding response history"
        if name == "DKT":
            std = model(q, s, r, a, m); first = torch.tensor([1, 3], device=dev)
            torch.testing.assert_close(std[0, first], base[0, first], rtol=0, atol=1e-7)
        report[name] = [float(x) for x in history_delta.cpu()]
    # The first canonical interaction is context only, even when rows are shuffled.
    seq = np.array([[0, 0, 0, 1, 0, 1], [0, 0, 0, 1, 0, 0],
                    [0, 0, 0, 1, 0, 2], [0, 0, 0, 1, 0, 1]], dtype=np.int32)
    sample = KTDataset({0: seq}, {0: [(0, 1, 2)]}, [0])[0]
    assert sample[-1].tolist() == [1, 0, 1, 1]
    RESULTS["protocol_tests"] = {"passed": True, "target_label_max_delta": max_target_delta,
                                 "preceding_label_delta_by_sibling": report,
                                 "canonical_first_interaction_excluded": True}
    print("Protocol smoke tests passed; preceding-label deltas by sibling:", report)

run_protocol_smoke_tests()

@torch.no_grad()
def eval_model(model, loader, ema=None, allinone=False):
    """Returns dict(row=..., fused=...). allinone=True: exact all-in-one
    protocol (faithful past context, target interaction's labels severed)."""
    model.eval()
    bm = model.module if isinstance(model, nn.DataParallel) else model
    if ema: ema.apply(bm)
    us, iis, ss, ys, ps = [], [], [], [], []
    for u, q, s, r, a, ii, m, target in loader:
        q, s, r, a, m, target = (x.to(DEVICE) for x in (q, s, r, a, m, target))
        ii_d = ii.to(DEVICE)
        if allinone and hasattr(bm, "allinone_logits"):
            logits = bm.allinone_logits(q, s, r, a, m, ii_d)
        elif allinone:
            logits = bm(q, s, r, a, m, allinone_ban(ii_d))
        else:
            logits = model(q, s, r, a, m)
        valid = (m == 1) & target
        vc = valid.cpu()
        us.append(u[vc].numpy()); iis.append(ii[vc].numpy()); ss.append(s[valid].cpu().numpy())
        ys.append(r[valid].cpu().numpy())
        ps.append(torch.sigmoid(logits[valid]).float().cpu().numpy())
    if ema: ema.restore(bm)
    if not ys: return {"row": float("nan"), "fused": float("nan"),
                       "n_rows": 0, "n_kcs": 0, "n_interactions": 0}
    u = np.concatenate(us); ii = np.concatenate(iis); s = np.concatenate(ss)
    y = np.concatenate(ys); p = np.concatenate(ps)
    raw = pd.DataFrame({"u": u, "iid": ii, "s": s, "y": y, "p": p})
    if raw.groupby(["u", "iid"])["y"].nunique().max() > 1:
        raise AssertionError("an interaction has inconsistent correctness labels")
    kc = raw.groupby(["u", "iid", "s"], sort=False).agg(y=("y", "first"), p=("p", "mean")).reset_index()
    grp = kc.groupby(["u", "iid"], sort=False).agg(y=("y", "first"), p=("p", "mean"))
    return {"row": compute_auc(y, p), "fused": compute_auc(grp["y"], grp["p"]),
            "n_rows": int(len(raw)), "n_kcs": int(len(kc)), "n_interactions": int(len(grp))}

def train_model(model_name, seqs, students, nq, ns, seed, label, window_catalog):
    """Trains with teacher forcing; selects on validation ALL-IN-ONE fused
    AUC (leak-free, uniform across conditions); reports test metrics under
    both protocols."""
    seed_everything(seed)
    trl, val, tel = make_loaders(seqs, students, nq, ns, seed, window_catalog)
    bm = build_model(model_name, nq, ns).to(DEVICE)
    model = nn.DataParallel(bm) if (DEVICE == "cuda" and torch.cuda.device_count() > 1) else bm
    n_par = sum(p.numel() for p in bm.parameters() if p.requires_grad)
    print(f"    [{label}] params={n_par:,} train_batches={len(trl)}")
    opt = torch.optim.AdamW(bm.parameters(), lr=CFG["lr"], weight_decay=CFG["wd"])
    sched = cosine_lr(opt, CFG["warmup"] * len(trl), CFG["epochs"] * len(trl))
    scaler = _scaler(CFG["use_amp"] and DEVICE == "cuda")
    ema = EMA(bm, CFG["ema_decay"])
    crit = crit_fn(CFG["label_smooth"])
    best, pat, best_state, best_shadow = -1., 0, None, None
    hist = {"epoch": [], "loss": [], "val_row": [], "val_ai_fused": []}
    for ep in range(1, CFG["epochs"] + 1):
        model.train(); ls = nb = 0
        for u, q, s, r, a, ii, m, target in trl:
            q, s, r, a, m, target = (x.to(DEVICE) for x in (q, s, r, a, m, target))
            opt.zero_grad(set_to_none=True)
            with _autocast(CFG["use_amp"] and DEVICE == "cuda"):
                logits = model(q, s, r, a, m)
                valid = (m == 1) & target
                loss = crit(logits, r)[valid].mean()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(bm.parameters(), CFG["clip"])
            scaler.step(opt); scaler.update(); sched.step(); ema.update(bm)
            ls += loss.item(); nb += 1
        va_std = eval_model(model, val, ema, allinone=False)
        va_ai  = eval_model(model, val, ema, allinone=True)
        hist["epoch"].append(ep); hist["loss"].append(ls / max(1, nb))
        hist["val_row"].append(va_std["row"]); hist["val_ai_fused"].append(va_ai["fused"])
        crit_metric = va_ai["fused"]
        if not math.isnan(crit_metric) and crit_metric > best:
            best = crit_metric; pat = 0
            best_state = {k: v.detach().cpu().clone() for k, v in bm.state_dict().items()}
            best_shadow = {k: v.clone() for k, v in ema.s.items()}
        else:
            pat += 1
        if ep % 5 == 0 or pat == 0:
            print(f"    ep{ep:3d} loss {ls/max(1,nb):.4f} | val row {va_std['row']:.4f} "
                  f"ai-fused {va_ai['fused']:.4f} | best {best:.4f} | pat {pat}")
        if pat >= CFG["early_stop"]:
            print(f"    early stop at ep{ep}"); break
    if best_state: bm.load_state_dict(best_state); ema.s = best_shadow
    te  = eval_model(model, tel, ema, allinone=False)
    tea = eval_model(model, tel, ema, allinone=True)
    print(f"    -> TEST row={te['row']:.4f} fused={te['fused']:.4f} | "
          f"all-in-one fused={tea['fused']:.4f}")
    del model, bm
    if DEVICE == "cuda": torch.cuda.empty_cache()
    return {"test_row": te["row"], "test_fused": te["fused"],
            "test_ai_row": tea["row"], "test_ai_fused": tea["fused"],
            "test_n_rows": te["n_rows"], "test_n_kcs": te["n_kcs"],
            "test_n_interactions": te["n_interactions"],
            "best_val_ai": best, "history": hist}

Protocol smoke tests passed; preceding-label deltas by sibling: {'DKT': [0.0070976316928863525, 0.0036666393280029297], 'AKT-R': [0.12752556800842285, 0.1205371618270874], 'simpleKT': [0.4886300563812256, 0.4815745949745178]}


In [7]:
# ================================================================
# 6. SECTIONS C–E — all Algebra2005 training runs (resumable)
#   C: DKT / AKT-R / simpleKT × {QL, EXP} × 5 seeds = 30 runs
#   D: simpleKT × SHUF × 5 seeds                     =  5 runs
#   E: simpleKT × LEGACY × 5 seeds                   =  5 runs
# ================================================================
ALG_INDEX = {student: index for index, student in enumerate(ALG_STUDENTS)}
ALG_WINDOWS = build_window_catalog(ALG_CONDS, ALG_INDEX, "ALG05")
RESULTS["stats"].setdefault("ALG05", dict(ALG_STATS))["window_catalog"] = ALG_WINDOWS["audit"]

RESULTS.setdefault("splits", {})["ALG05"] = {}
for seed in SEEDS:
    train_idx, validation_idx, test_idx = split_students(ALG_STUDENTS, seed)
    RESULTS["splits"]["ALG05"][str(seed)] = {
        "train": [ALG_STUDENTS[i] for i in train_idx],
        "validation": [ALG_STUDENTS[i] for i in validation_idx],
        "test": [ALG_STUDENTS[i] for i in test_idx],
    }
save_results()

_PREP_CACHE = {}
def get_prepared(condition):
    if condition not in _PREP_CACHE:
        _PREP_CACHE[condition] = (
            prepare_condition(ALG_CONDS[condition], ALG_INDEX)
            + (ALG_STUDENTS, ALG_WINDOWS)
        )
    return _PREP_CACHE[condition]

RUN_PLAN = []
if RUN["C"]:
    for model in ("DKT", "AKT-R", "simpleKT"):
        for condition in ("QL", "EXP"):
            for seed in SEEDS:
                RUN_PLAN.append(("ALG05", model, condition, seed))
if RUN["D"]:
    for seed in SEEDS:
        RUN_PLAN.append(("ALG05", "simpleKT", "SHUF", seed))
if RUN["E"]:
    for seed in SEEDS:
        RUN_PLAN.append(("ALG05", "simpleKT", "LEGACY", seed))

todo = sum(1 for key in RUN_PLAN if "/".join(map(str, key)) not in RESULTS["runs"])
print(f"Run plan: {len(RUN_PLAN)} runs ({todo} still to do)\n")

for dataset, model_name, condition, seed in RUN_PLAN:
    key = f"{dataset}/{model_name}/{condition}/{seed}"
    if key in RESULTS["runs"]:
        result = RESULTS["runs"][key]
        print(f"[skip] {key} (row={result['test_row']:.4f}; ai-fused={result['test_ai_fused']:.4f})")
        continue
    print(f"[run ] {key}")
    start = time.time()
    seqs, nq, ns, students, windows = get_prepared(condition)
    result = train_model(model_name, seqs, students, nq, ns, seed, key, windows)
    result["minutes"] = round((time.time() - start) / 60, 1)
    RESULTS["runs"][key] = result
    save_results()
    print(f"       done in {result['minutes']} min\n")

print("All requested Algebra2005 runs finished.")


  [ALG05] shared windows: train={'windows': 5874, 'shortened_by_shared_row_budget': 1, 'max_shared_rows': 384, 'max_interactions': 200}, eval={'windows': 3343, 'shortened_by_shared_row_budget': 1, 'max_shared_rows': 384, 'max_interactions': 200}
Run plan: 40 runs (40 still to do)

[run ] ALG05/DKT/QL/42
    [ALG05/DKT/QL/42] params=133,722 train_batches=36
    ep  1 loss 0.6837 | val row 0.5259 ai-fused 0.5259 | best 0.5259 | pat 0
    ep  2 loss 0.5777 | val row 0.5650 ai-fused 0.5650 | best 0.5650 | pat 0
    ep  3 loss 0.5051 | val row 0.6359 ai-fused 0.6359 | best 0.6359 | pat 0
    ep  4 loss 0.4898 | val row 0.6925 ai-fused 0.6925 | best 0.6925 | pat 0
    ep  5 loss 0.4816 | val row 0.7285 ai-fused 0.7285 | best 0.7285 | pat 0
    ep  6 loss 0.4749 | val row 0.7518 ai-fused 0.7518 | best 0.7518 | pat 0
    ep  7 loss 0.4706 | val row 0.7663 ai-fused 0.7663 | best 0.7663 | pat 0
    ep  8 loss 0.4671 | val row 0.7755 ai-fused 0.7755 | best 0.7755 | pat 0
    ep  9 loss 0.4639 | v

In [8]:
# ================================================================
# 7. SECTION G — aggregate journal-replication results and figures
# ================================================================
from scipy import stats as _st

def runs(model, condition, metric="test_row"):
    return [RESULTS["runs"][f"ALG05/{model}/{condition}/{seed}"][metric]
            for seed in SEEDS if f"ALG05/{model}/{condition}/{seed}" in RESULTS["runs"]]

def mean_sd(values):
    if not values:
        return "--"
    return f"{np.mean(values):.4f}±{np.std(values, ddof=1):.4f}"

def paired(a, b):
    """a-b over identical seed/student splits."""
    if not a or not b or len(a) != len(b):
        return None
    difference = np.asarray(a) - np.asarray(b)
    n = len(difference)
    mean = float(difference.mean())
    sd = float(difference.std(ddof=1)) if n > 1 else 0.0
    half = float(_st.t.ppf(0.975, n - 1) * sd / np.sqrt(n)) if n > 1 else 0.0
    return {"mean": mean, "sd": sd, "lo": mean - half, "hi": mean + half, "n": n}

def paired_string(result):
    if result is None:
        return "--"
    return f"{result['mean']:+.4f} [{result['lo']:+.4f}, {result['hi']:+.4f}]"

if RUN["G"]:
    MODELS = ["DKT", "AKT-R", "simpleKT"]
    missing = ["/".join(map(str, key)) for key in RUN_PLAN
               if "/".join(map(str, key)) not in RESULTS["runs"]]
    if missing:
        raise RuntimeError(f"Aggregation requires every requested run; missing {missing[:5]}")

    # Target matching and QL protocol-equivalence checks.
    max_ql_protocol_delta = 0.0
    for seed in SEEDS:
        for model in MODELS:
            ql = RESULTS["runs"][f"ALG05/{model}/QL/{seed}"]
            exp = RESULTS["runs"][f"ALG05/{model}/EXP/{seed}"]
            assert ql["test_n_interactions"] == exp["test_n_interactions"], "QL/EXP target mismatch"
            max_ql_protocol_delta = max(
                max_ql_protocol_delta,
                abs(ql["test_row"] - ql["test_fused"]),
                abs(ql["test_row"] - ql["test_ai_fused"]),
            )
        exp = RESULTS["runs"][f"ALG05/simpleKT/EXP/{seed}"]
        shuf = RESULTS["runs"][f"ALG05/simpleKT/SHUF/{seed}"]
        legacy = RESULTS["runs"][f"ALG05/simpleKT/LEGACY/{seed}"]
        assert (exp["test_n_interactions"], exp["test_n_kcs"]) == (
            shuf["test_n_interactions"], shuf["test_n_kcs"]
        ), "EXP/SHUF fused-target mismatch"
        assert (exp["test_n_interactions"], exp["test_n_kcs"]) == (
            legacy["test_n_interactions"], legacy["test_n_kcs"]
        ), "EXP/LEGACY fused-target mismatch"
    assert max_ql_protocol_delta < 1e-7, max_ql_protocol_delta
    RESULTS["replication_tests"] = {
        "passed": True,
        "target_counts_matched": True,
        "max_ql_protocol_delta": max_ql_protocol_delta,
    }

    summary = {"main": {}, "sequential_protocol_contrasts": {},
               "order_disruption": {}, "source_decomposition": {}}
    print("=" * 112)
    print("TABLE A — Algebra2005 replication: training representation × evaluation protocol")
    print(f"{'':10s} {'QL ai-fused':>16s} {'EXP row':>16s} {'EXP fused':>16s} {'EXP ai-fused':>16s} "
          f"{'row−QL [95% CI]':>26s} {'ai−QL [95% CI]':>26s}")
    for model in MODELS:
        ql_ai = runs(model, "QL", "test_ai_fused")
        exp_row = runs(model, "EXP", "test_row")
        exp_fused = runs(model, "EXP", "test_fused")
        exp_ai = runs(model, "EXP", "test_ai_fused")
        row_gap = paired(exp_row, ql_ai)
        ai_gap = paired(exp_ai, ql_ai)
        summary["main"][model] = {
            "ql_ai_fused": ql_ai, "exp_row": exp_row,
            "exp_fused": exp_fused, "exp_ai_fused": exp_ai,
            "exp_row_minus_ql": row_gap,
            "exp_ai_minus_ql": ai_gap,
        }
        print(f"{model:10s} {mean_sd(ql_ai):>16s} {mean_sd(exp_row):>16s} "
              f"{mean_sd(exp_fused):>16s} {mean_sd(exp_ai):>16s} "
              f"{paired_string(row_gap):>26s} {paired_string(ai_gap):>26s}")

    print("\n" + "=" * 112)
    print("TABLE B — Sequential evaluation-protocol contrasts (EXP-trained)")
    print(f"{'':10s} {'row−fused [95% CI]':>27s} {'fused−ai [95% CI]':>27s} "
          f"{'ai−QL [95% CI]':>27s}")
    for model in MODELS:
        row = runs(model, "EXP", "test_row")
        fused = runs(model, "EXP", "test_fused")
        ai = runs(model, "EXP", "test_ai_fused")
        ql = runs(model, "QL", "test_ai_fused")
        contrasts = {
            "row_minus_fused": paired(row, fused),
            "fused_minus_ai": paired(fused, ai),
            "ai_minus_ql": paired(ai, ql),
        }
        summary["sequential_protocol_contrasts"][model] = contrasts
        print(f"{model:10s} {paired_string(contrasts['row_minus_fused']):>27s} "
              f"{paired_string(contrasts['fused_minus_ai']):>27s} "
              f"{paired_string(contrasts['ai_minus_ql']):>27s}")

    print("\n" + "=" * 112)
    print("TABLE C — Order-disruption probe (simpleKT, row-level evaluation)")
    ql = runs("simpleKT", "QL")
    exp = runs("simpleKT", "EXP")
    shuf = runs("simpleKT", "SHUF")
    summary["order_disruption"] = {
        "ql": ql, "exp": exp, "shuf": shuf,
        "exp_minus_shuf": paired(exp, shuf),
        "shuf_minus_ql": paired(shuf, ql),
    }
    print(f"  QL                 {mean_sd(ql)}")
    print(f"  EXP consecutive    {mean_sd(exp)}")
    print(f"  SHUF disrupted     {mean_sd(shuf)}")
    print(f"  EXP−SHUF           {paired_string(summary['order_disruption']['exp_minus_shuf'])}")
    print(f"  SHUF−QL            {paired_string(summary['order_disruption']['shuf_minus_ql'])}")

    print("\n" + "=" * 112)
    print("TABLE D — Source-decomposition replication (simpleKT, row-level evaluation)")
    legacy = runs("simpleKT", "LEGACY")
    summary["source_decomposition"] = {
        "ql": ql,
        "exp": exp,
        "legacy": legacy,
        "exp_minus_ql": paired(exp, ql),
        "legacy_minus_exp": paired(legacy, exp),
        "raw_source_duplicate_excess_rows": ALG_STATS["source_duplicate_excess_rows"],
    }
    print(f"  QL                 {mean_sd(ql)}")
    print(f"  EXP                {mean_sd(exp)}")
    print(f"  LEGACY             {mean_sd(legacy)}")
    print(f"  EXP−QL             {paired_string(summary['source_decomposition']['exp_minus_ql'])}")
    print(f"  LEGACY−EXP         {paired_string(summary['source_decomposition']['legacy_minus_exp'])}")

    RESULTS["journal_replication_summary"] = summary
    save_results()

    # Figure 3: main training × evaluation comparison.
    fig, ax = plt.subplots(figsize=(11, 6))
    x = np.arange(len(MODELS)); width = 0.25
    series = [
        (-width, "EXP", "test_row", "#F44336", "EXP-trained, row-level"),
        (0.0, "EXP", "test_ai_fused", "#FF9800", "EXP-trained, all-in-one"),
        (width, "QL", "test_ai_fused", "#1976D2", "QL-trained, all-in-one"),
    ]
    for offset, condition, metric, color, label in series:
        means = [np.mean(runs(model, condition, metric)) for model in MODELS]
        errors = [np.std(runs(model, condition, metric), ddof=1) for model in MODELS]
        bars = ax.bar(x + offset, means, width, yerr=errors, capsize=4,
                      color=color, alpha=0.9, label=label)
        for bar, value in zip(bars, means):
            ax.text(bar.get_x() + bar.get_width()/2, value + 0.006, f"{value:.3f}",
                    ha="center", fontsize=8, fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(MODELS); ax.set_ylabel("test AUC")
    ax.axhline(0.5, color="gray", ls="--", lw=0.8)
    ax.set_ylim(0.45, 1.0)
    ax.set_title("Algebra2005: training representation × evaluation protocol\nmean ± sample sd, 5 paired seeds")
    ax.legend(fontsize=9); plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/alg_fig3_training_by_evaluation.png", dpi=150, bbox_inches="tight")
    plt.close()

    # Figure 4: all sequential protocol scores.
    fig, ax = plt.subplots(figsize=(11, 6))
    x = np.arange(len(MODELS)); width = 0.2
    series = [
        ("QL-trained (all-in-one)", [runs(m, "QL", "test_ai_fused") for m in MODELS], "#1976D2"),
        ("EXP all-in-one", [runs(m, "EXP", "test_ai_fused") for m in MODELS], "#4CAF50"),
        ("EXP fused", [runs(m, "EXP", "test_fused") for m in MODELS], "#FF9800"),
        ("EXP row", [runs(m, "EXP", "test_row") for m in MODELS], "#F44336"),
    ]
    for i, (label, values, color) in enumerate(series):
        offset = (i - 1.5) * width
        means = [np.mean(v) for v in values]
        errors = [np.std(v, ddof=1) for v in values]
        bars = ax.bar(x + offset, means, width, yerr=errors, capsize=4,
                      color=color, alpha=0.9, label=label)
        for bar, value in zip(bars, means):
            ax.text(bar.get_x() + bar.get_width()/2, value + 0.006, f"{value:.3f}",
                    ha="center", fontsize=7, fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(MODELS); ax.set_ylabel("test AUC")
    ax.axhline(0.5, color="gray", ls="--", lw=0.8)
    ax.set_ylim(0.45, 1.0)
    ax.set_title("Algebra2005 evaluation-protocol contrasts\nmean ± sample sd, 5 paired seeds")
    ax.legend(fontsize=9); plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/alg_fig4_protocol_contrasts.png", dpi=150, bbox_inches="tight")
    plt.close()

    # Figure 5: order-disruption test and diagnostic validation curves.
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    labels = ["QL", "EXP\nconsecutive", "SHUF\ndisrupted"]
    values = [ql, exp, shuf]
    means = [np.mean(v) for v in values]
    errors = [np.std(v, ddof=1) for v in values]
    bars = axes[0].bar(labels, means, yerr=errors, capsize=4,
                       color=["#1976D2", "#F44336", "#FF9800"])
    for bar, value in zip(bars, means):
        axes[0].text(bar.get_x() + bar.get_width()/2, value + 0.006, f"{value:.4f}",
                     ha="center", fontsize=9, fontweight="bold")
    axes[0].set_ylabel("test AUC (row-level)")
    axes[0].axhline(0.5, color="gray", ls="--", lw=0.8)
    axes[0].set_ylim(0.45, 1.0)
    axes[0].set_title("Order-disruption probe")
    for condition, label, color, line in [
        ("QL", "QL", "#1976D2", "-"),
        ("EXP", "EXP consecutive", "#F44336", "-"),
        ("SHUF", "SHUF disrupted", "#FF9800", "--"),
    ]:
        key = f"ALG05/simpleKT/{condition}/{MAIN_SEED}"
        history = RESULTS["runs"][key]["history"]
        axes[1].plot(history["epoch"], history["val_row"], line, color=color, label=label)
    axes[1].set_xlabel("epoch"); axes[1].set_ylabel("validation row AUC (diagnostic)")
    axes[1].set_title("Seed 42 curves; selection used all-in-one AUC")
    axes[1].legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/alg_fig5_order_disruption.png", dpi=150, bbox_inches="tight")
    plt.close()

    # Figure 6: source decomposition retained from the ASSIST2009 protocol.
    fig, ax = plt.subplots(figsize=(8, 5))
    labels = ["QL", "EXP", "LEGACY"]
    values = [ql, exp, legacy]
    means = [np.mean(v) for v in values]
    errors = [np.std(v, ddof=1) for v in values]
    bars = ax.bar(labels, means, yerr=errors, capsize=5,
                  color=["#1976D2", "#FF9800", "#B71C1C"])
    for bar, value in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, value + 0.006, f"{value:.4f}",
                ha="center", fontsize=10, fontweight="bold")
    ax.axhline(0.5, color="gray", ls="--", lw=0.8)
    ax.set_ylim(0.45, 1.0)
    ax.set_ylabel("test AUC (row-level, simpleKT)")
    ax.set_title(
        "Algebra2005 source decomposition\n"
        f"raw source-duplicate excess: {ALG_STATS['source_duplicate_excess_rows']:,} rows"
    )
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/alg_fig6_source_decomposition.png", dpi=150, bbox_inches="tight")
    plt.close()

    import shutil
    # Keep the handoff name used by the manuscript-integration workflow.
    archive = shutil.make_archive("paper_alg_results", "zip", OUT_DIR)
    print(f"\nSaved results and figures in {OUT_DIR}/")
    print(f"Download: {archive}")


TABLE A — Algebra2005 replication: training representation × evaluation protocol
                QL ai-fused          EXP row        EXP fused     EXP ai-fused            row−QL [95% CI]             ai−QL [95% CI]
DKT           0.8251±0.0057    0.9220±0.0031    0.8943±0.0044    0.7983±0.0074 +0.0969 [+0.0914, +0.1024] -0.0268 [-0.0311, -0.0226]
AKT-R         0.8044±0.0083    0.9074±0.0055    0.8733±0.0073    0.7724±0.0047 +0.1030 [+0.0925, +0.1135] -0.0321 [-0.0403, -0.0238]
simpleKT      0.8070±0.0070    0.8471±0.0051    0.8322±0.0063    0.8080±0.0059 +0.0401 [+0.0317, +0.0485] +0.0010 [-0.0020, +0.0039]

TABLE B — Sequential evaluation-protocol contrasts (EXP-trained)
                    row−fused [95% CI]           fused−ai [95% CI]              ai−QL [95% CI]
DKT         +0.0277 [+0.0256, +0.0298]  +0.0960 [+0.0876, +0.1044]  -0.0268 [-0.0311, -0.0226]
AKT-R       +0.0341 [+0.0316, +0.0366]  +0.1009 [+0.0916, +0.1103]  -0.0321 [-0.0403, -0.0238]
simpleKT    +0.0149 [+0.0107, +0.019

## After the run

Download **`paper_alg_results.zip`**. It contains:

- `algebra2005_replication_results.json` with the raw-file hash, filters, representation tests,
  exact student splits, shared-window audit, protocol smoke tests, training histories, per-seed
  metrics, paired contrasts, and manuscript-ready summary values;
- `alg_fig1`–`alg_fig6` generated from that same JSON-backed run.

The decisive journal comparison is **EXP all-in-one − QL all-in-one**, paired by the same five
seeds and student splits. The row-level EXP − QL contrast is retained as the replication of the
apparent leaky advantage. `row − fused` and `fused − all-in-one` remain sequential protocol
contrasts rather than additive causal components.

The **LEGACY − EXP** paired contrast reproduces the ASSIST2009 source-decomposition step.
If the raw Algebra file contains no repeated KC tokens within a source interaction, LEGACY and
EXP are identical and this contrast should be exactly zero; that is a protocol result, not a
missing analysis.

If Kaggle stops before all 40 runs finish, attach the partial JSON to a new session, copy it to
`/kaggle/working/algebra2005_replication_out/algebra2005_replication_results.json`, and rerun.
Compatible completed runs are skipped automatically.


In [9]:
# ================================================================
# TEST 4 - INTERACTION-AWARE, LEAK-FREE EXP TRAINING
# Paste this cell AFTER the completed Algebra2005 notebook cells.
#
# Scientific contrast:
#   conventional EXP training -> teacher-forced sibling-label paths
#   EXP-AIT training          -> those paths are removed during training
#
# Everything else is held fixed: EXP data, student splits, shared windows,
# architecture, initialization seeds, optimizer, hyperparameters, EMA,
# checkpoint objective, and exact all-in-one test evaluation.
#
# The loss remains row-weighted deliberately. Thus EXP-AIT versus EXP changes
# only the forward information paths, not the training target weights.
# ================================================================
import os, time, math, shutil
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from scipy import stats as _ia_stats

IA_MODELS = ["DKT", "AKT-R", "simpleKT"]
IA_SEEDS = list(SEEDS)
IA_CONDITION = "EXP-AIT"
IA_FORCE_RERUN = False

_required = [
    "ALG_CONDS", "ALG_INDEX", "ALG_WINDOWS", "ALG_STUDENTS", "RESULTS",
    "prepare_condition", "make_loaders", "build_model", "allinone_ban",
    "eval_model", "EMA", "crit_fn", "cosine_lr", "save_results",
]
_missing = [name for name in _required if name not in globals()]
if _missing:
    raise RuntimeError(
        "Run the Algebra2005 notebook through its aggregation cell first. "
        f"Missing objects: {_missing}"
    )


class _InteractionAwareForward(nn.Module):
    """Expose exact all-in-one prediction as forward() for training.

    DKT: every sibling is predicted from the recurrent state at the end of
    the previous interaction. Attention models: all keys/values belonging to
    the query's interaction are banned; queries contain no response label.
    """
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, q, s, r, a, mask, interaction_id):
        if hasattr(self.base_model, "allinone_logits"):
            return self.base_model.allinone_logits(q, s, r, a, mask, interaction_id)
        return self.base_model(q, s, r, a, mask, allinone_ban(interaction_id))


def _train_model_interaction_aware(
    model_name, seqs, students, nq, ns, seed, label, window_catalog
):
    """Train EXP from scratch with exact all-in-one information restrictions.

    Checkpoint selection is the same leak-free validation all-in-one fused AUC
    used by every original run. BCE is averaged over target rows, matching the
    conventional EXP loss and isolating the training-time information change.
    """
    seed_everything(seed)
    train_loader, validation_loader, test_loader = make_loaders(
        seqs, students, nq, ns, seed, window_catalog
    )

    base_model = build_model(model_name, nq, ns).to(DEVICE)
    ia_forward = _InteractionAwareForward(base_model).to(DEVICE)
    train_model = (
        nn.DataParallel(ia_forward)
        if DEVICE == "cuda" and torch.cuda.device_count() > 1
        else ia_forward
    )

    n_parameters = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
    print(f"    [{label}] params={n_parameters:,} train_batches={len(train_loader)}")

    optimizer = torch.optim.AdamW(
        base_model.parameters(), lr=CFG["lr"], weight_decay=CFG["wd"]
    )
    scheduler = cosine_lr(
        optimizer,
        CFG["warmup"] * len(train_loader),
        CFG["epochs"] * len(train_loader),
    )
    scaler = _scaler(CFG["use_amp"] and DEVICE == "cuda")
    ema = EMA(base_model, CFG["ema_decay"])
    criterion = crit_fn(CFG["label_smooth"])

    best = -1.0
    patience = 0
    best_state = None
    best_shadow = None
    history = {"epoch": [], "loss": [], "val_ai_fused": []}

    for epoch in range(1, CFG["epochs"] + 1):
        train_model.train()
        loss_sum = 0.0
        n_batches = 0

        for u, q, s, r, a, interaction_id, mask, target in train_loader:
            q, s, r, a, interaction_id, mask, target = (
                x.to(DEVICE)
                for x in (q, s, r, a, interaction_id, mask, target)
            )
            optimizer.zero_grad(set_to_none=True)

            with _autocast(CFG["use_amp"] and DEVICE == "cuda"):
                logits = train_model(q, s, r, a, mask, interaction_id)
                valid = (mask == 1) & target
                loss = criterion(logits, r)[valid].mean()

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(base_model.parameters(), CFG["clip"])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            ema.update(base_model)

            loss_sum += float(loss.item())
            n_batches += 1

        # Exact same checkpoint objective as the original notebook.
        validation_ai = eval_model(
            base_model, validation_loader, ema, allinone=True
        )
        validation_score = validation_ai["fused"]
        mean_loss = loss_sum / max(1, n_batches)

        history["epoch"].append(epoch)
        history["loss"].append(mean_loss)
        history["val_ai_fused"].append(validation_score)

        if not math.isnan(validation_score) and validation_score > best:
            best = validation_score
            patience = 0
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in base_model.state_dict().items()
            }
            best_shadow = {key: value.clone() for key, value in ema.s.items()}
        else:
            patience += 1

        if epoch % 5 == 0 or patience == 0:
            print(
                f"    ep{epoch:3d} loss {mean_loss:.4f} | "
                f"val ai-fused {validation_score:.4f} | "
                f"best {best:.4f} | pat {patience}"
            )
        if patience >= CFG["early_stop"]:
            print(f"    early stop at ep{epoch}")
            break

    if best_state is None:
        raise RuntimeError(f"{label}: no valid checkpoint was produced")

    base_model.load_state_dict(best_state)
    ema.s = best_shadow

    # Standard scores are diagnostic. The preregistered comparison below uses
    # exact all-in-one fused AUC.
    test_standard = eval_model(base_model, test_loader, ema, allinone=False)
    test_ai = eval_model(base_model, test_loader, ema, allinone=True)
    print(
        f"    -> TEST row={test_standard['row']:.4f} "
        f"fused={test_standard['fused']:.4f} | "
        f"all-in-one fused={test_ai['fused']:.4f}"
    )

    result = {
        "training_protocol": "exact all-in-one forward; row-weighted smoothed BCE",
        "test_row": test_standard["row"],
        "test_fused": test_standard["fused"],
        "test_ai_row": test_ai["row"],
        "test_ai_fused": test_ai["fused"],
        "test_n_rows": test_standard["n_rows"],
        "test_n_kcs": test_standard["n_kcs"],
        "test_n_interactions": test_standard["n_interactions"],
        "best_val_ai": best,
        "history": history,
    }

    del train_model, ia_forward, base_model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return result


def _paired_summary(a, b):
    """Paired a-b t interval over identical seeds and student splits."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if len(a) != len(b) or len(a) == 0:
        raise ValueError("Paired samples must be non-empty and have equal length")
    differences = a - b
    n = len(differences)
    mean = float(differences.mean())
    sd = float(differences.std(ddof=1)) if n > 1 else 0.0
    half_width = (
        float(_ia_stats.t.ppf(0.975, n - 1) * sd / np.sqrt(n))
        if n > 1
        else 0.0
    )
    return {
        "mean": mean,
        "sd": sd,
        "lo": mean - half_width,
        "hi": mean + half_width,
        "n": n,
        "per_seed": differences.tolist(),
    }


# Prepare exactly the existing EXP representation and shared window catalog.
_ia_seqs, _ia_nq, _ia_ns = prepare_condition(ALG_CONDS["EXP"], ALG_INDEX)

RESULTS.setdefault("interaction_aware_training", {})["design"] = {
    "condition": IA_CONDITION,
    "dataset": "ALG05",
    "training_representation": "EXP",
    "information_protocol": "exact all-in-one during training",
    "loss_weighting": "row-weighted, matching conventional EXP",
    "checkpoint_objective": "validation all-in-one fused AUC",
    "primary_test_metric": "test all-in-one fused AUC",
    "paired_seeds": list(IA_SEEDS),
    "only_intended_change_vs_EXP": "training-time same-interaction label paths removed",
}
save_results()

_ia_plan = [
    (model_name, seed)
    for model_name in IA_MODELS
    for seed in IA_SEEDS
]
_ia_todo = sum(
    IA_FORCE_RERUN
    or f"ALG05/{model_name}/{IA_CONDITION}/{seed}" not in RESULTS["runs"]
    for model_name, seed in _ia_plan
)
print(
    f"\nTest 4 plan: {len(_ia_plan)} runs "
    f"({_ia_todo} still to do; resumable after each run)\n"
)

for _ia_model_name, _ia_seed in _ia_plan:
    _ia_key = f"ALG05/{_ia_model_name}/{IA_CONDITION}/{_ia_seed}"
    if _ia_key in RESULTS["runs"] and not IA_FORCE_RERUN:
        _ia_result = RESULTS["runs"][_ia_key]
        print(
            f"[skip] {_ia_key} "
            f"(ai-fused={_ia_result['test_ai_fused']:.4f})"
        )
        continue

    print(f"[run ] {_ia_key}")
    _ia_start = time.time()
    _ia_result = _train_model_interaction_aware(
        _ia_model_name,
        _ia_seqs,
        ALG_STUDENTS,
        _ia_nq,
        _ia_ns,
        _ia_seed,
        _ia_key,
        ALG_WINDOWS,
    )
    _ia_result["minutes"] = round((time.time() - _ia_start) / 60, 1)

    # Strong target-matching assertion against the same seed's EXP run.
    _exp_key = f"ALG05/{_ia_model_name}/EXP/{_ia_seed}"
    if _exp_key not in RESULTS["runs"]:
        raise RuntimeError(f"Missing conventional comparison run: {_exp_key}")
    _exp_result = RESULTS["runs"][_exp_key]
    assert (
        _ia_result["test_n_rows"],
        _ia_result["test_n_kcs"],
        _ia_result["test_n_interactions"],
    ) == (
        _exp_result["test_n_rows"],
        _exp_result["test_n_kcs"],
        _exp_result["test_n_interactions"],
    ), f"{_ia_key}: target counts do not match conventional EXP"

    RESULTS["runs"][_ia_key] = _ia_result
    save_results()
    print(f"       done in {_ia_result['minutes']} min\n")


def _values(model_name, condition, metric="test_ai_fused"):
    return [
        RESULTS["runs"][f"ALG05/{model_name}/{condition}/{seed}"][metric]
        for seed in IA_SEEDS
    ]


# Primary paired analysis. Do not interpret the standard row score as fair.
_ia_summary = {}
print("=" * 108)
print("TEST 4 - interaction-aware EXP training; fair test metric = all-in-one fused AUC")
print(
    f"{'model':10s} {'QL mean+/-sd':>16s} {'EXP mean+/-sd':>16s} "
    f"{'EXP-AIT mean+/-sd':>19s} {'AIT-EXP [95% CI]':>23s} "
    f"{'AIT-QL [95% CI]':>23s}"
)

for _ia_model_name in IA_MODELS:
    _ql = _values(_ia_model_name, "QL")
    _exp = _values(_ia_model_name, "EXP")
    _ait = _values(_ia_model_name, IA_CONDITION)
    _ait_minus_exp = _paired_summary(_ait, _exp)
    _ait_minus_ql = _paired_summary(_ait, _ql)
    _ia_summary[_ia_model_name] = {
        "ql_ai_fused": _ql,
        "exp_ai_fused": _exp,
        "exp_ait_ai_fused": _ait,
        "exp_ait_minus_exp": _ait_minus_exp,
        "exp_ait_minus_ql": _ait_minus_ql,
    }

    def _msd(values):
        return f"{np.mean(values):.4f}+/-{np.std(values, ddof=1):.4f}"

    def _pci(result):
        return (
            f"{result['mean']:+.4f} "
            f"[{result['lo']:+.4f}, {result['hi']:+.4f}]"
        )

    print(
        f"{_ia_model_name:10s} {_msd(_ql):>16s} {_msd(_exp):>16s} "
        f"{_msd(_ait):>19s} {_pci(_ait_minus_exp):>23s} "
        f"{_pci(_ait_minus_ql):>23s}"
    )

RESULTS["interaction_aware_training"]["summary"] = _ia_summary
RESULTS["interaction_aware_training"]["tests"] = {
    "completed_runs": len(_ia_plan),
    "target_counts_match_conventional_exp": True,
}
save_results()


# Manuscript-ready figure generated from the saved per-seed results.
fig, ax = plt.subplots(figsize=(11, 6))
_x = np.arange(len(IA_MODELS))
_width = 0.24
_series = [
    (-_width, "QL", "#1976D2", "QL-trained"),
    (0.0, "EXP", "#F57C00", "EXP-trained (conventional)"),
    (_width, IA_CONDITION, "#2E7D32", "EXP-trained (interaction-aware)"),
]
for _offset, _condition, _color, _label in _series:
    _all_values = [_values(model, _condition) for model in IA_MODELS]
    _means = [float(np.mean(values)) for values in _all_values]
    _errors = [float(np.std(values, ddof=1)) for values in _all_values]
    _bars = ax.bar(
        _x + _offset,
        _means,
        _width,
        yerr=_errors,
        capsize=4,
        color=_color,
        alpha=0.9,
        label=_label,
    )
    for _bar, _value in zip(_bars, _means):
        ax.text(
            _bar.get_x() + _bar.get_width() / 2,
            _value + 0.006,
            f"{_value:.3f}",
            ha="center",
            fontsize=8,
            fontweight="bold",
        )

ax.set_xticks(_x)
ax.set_xticklabels(IA_MODELS)
ax.set_ylabel("test AUC (all-in-one, interaction-fused)")
ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
ax.set_ylim(0.45, 1.0)
ax.set_title(
    "Algebra2005: interaction-aware EXP training\n"
    "mean +/- sample SD over five paired student splits"
)
ax.legend(fontsize=9)
plt.tight_layout()
_ia_figure_path = os.path.join(OUT_DIR, "alg_fig7_interaction_aware_training.png")
plt.savefig(_ia_figure_path, dpi=150, bbox_inches="tight")
plt.close().

# Refresh the same downloadable handoff archive used by the notebook.
_ia_archive = shutil.make_archive("paper_alg_results", "zip", OUT_DIR)
print(f"\nSaved: {_ia_figure_path}")
print(f"Updated JSON: {RESULTS_PATH}")
print(f"Download: {_ia_archive}")



Test 4 plan: 15 runs (15 still to do; resumable after each run)

[run ] ALG05/DKT/EXP-AIT/42
    [ALG05/DKT/EXP-AIT/42] params=142,577 train_batches=36
    ep  1 loss 0.6876 | val ai-fused 0.4640 | best 0.4640 | pat 0
    ep  2 loss 0.5966 | val ai-fused 0.5290 | best 0.5290 | pat 0
    ep  3 loss 0.5282 | val ai-fused 0.6157 | best 0.6157 | pat 0
    ep  4 loss 0.5074 | val ai-fused 0.6896 | best 0.6896 | pat 0
    ep  5 loss 0.4989 | val ai-fused 0.7299 | best 0.7299 | pat 0
    ep  6 loss 0.4922 | val ai-fused 0.7525 | best 0.7525 | pat 0
    ep  7 loss 0.4876 | val ai-fused 0.7655 | best 0.7655 | pat 0
    ep  8 loss 0.4844 | val ai-fused 0.7736 | best 0.7736 | pat 0
    ep  9 loss 0.4819 | val ai-fused 0.7792 | best 0.7792 | pat 0
    ep 10 loss 0.4795 | val ai-fused 0.7834 | best 0.7834 | pat 0
    ep 11 loss 0.4777 | val ai-fused 0.7869 | best 0.7869 | pat 0
    ep 12 loss 0.4757 | val ai-fused 0.7900 | best 0.7900 | pat 0
    ep 13 loss 0.4742 | val ai-fused 0.7928 | best 0.79